# 04 — Plug in your own growth law

Companion to
[`examples/custom_simulation.py`](../examples/custom_simulation.py).
`grow_tree` exposes four customisation hooks beyond the YAML config:

| keyword | type | what it controls |
| --- | --- | --- |
| `wind_fn` | `(generation, rng) -> (x, y, z)` | per-step wind vector |
| `sun` | `mechatree.light.Sun` | hemispherical light grid |
| `safety` | `mechatree.genome.SafetyModel` | mechanical headroom rule |
| `allocation` | `mechatree.genome.AllocationModel` | resource splits |

When a hook is omitted, `grow_tree` falls back to the Fortran-faithful
defaults: a rotating storm with a long-tailed amplitude, a 4×8
elevation/azimuth sun, and the constants from the YAML's `genome:`
block.

In [ ]:
import math
from pathlib import Path

import numpy as np
from plotly.subplots import make_subplots

from mechatree.config import load_config
from mechatree.genome import ConstantAllocation, ConstantSafety
from mechatree.light import Sun
from mechatree.plotting import plot_tree_3d
from mechatree.simulate import grow_tree

cfg = load_config(Path("../examples/forest.yaml").resolve())

## Two custom wind functions

A `wind_fn` is a plain Python callable. The default mirrors
`mod_evolu.f90`'s rotating storm — direction sweeps the compass while
amplitude follows a long-tailed distribution. Here are two simpler
alternatives:

In [ ]:
def steady_west_wind(generation: int, rng: np.random.Generator):
    """A trade-wind stand-in: same direction, same amplitude, every step."""
    return (1.0, 0.0, 0.0)


def storm_at_gen_50(generation: int, rng: np.random.Generator):
    """Calm for the first 50 generations, then a fixed southerly gale."""
    if generation < 50:
        return (0.0, 0.0, 0.0)
    return (0.0, 3.0, 0.0)

## Three scenarios

1. **Default** — Fortran storm wind, default sun, default constant genome.
2. **Steady-west + overhead sun + cautious genome** — combines every
   knob. `Sun.from_arrays` lets us pick arbitrary elevation/azimuth
   pairs rather than the regular grid. A lower safety factor (2.0
   instead of 3.0) means branches sit closer to their breaking stress,
   so the tree saves on wood; pushing all resources into leaves
   (`p_seeds=0.0`) makes it a long-lived, sterile sapling.
3. **Calm-then-storm** — quiet growth followed by a single decisive
   event at gen 50.

Same seed across all three, so the only thing that changes is the rule.

In [ ]:
n_generations = 80
seed = 42

t_default = grow_tree(cfg, n_generations=n_generations, seed=seed)

overhead_sun = Sun.from_arrays(elev=[0.1, 0.4], azim=[0.0, math.pi])
cautious_safety = ConstantSafety(2.0)
save_reserves = ConstantAllocation(p_seeds=0.0, p_leaves=0.3, phototropism=0.5)
t_steady = grow_tree(
    cfg,
    n_generations=n_generations,
    seed=seed,
    safety=cautious_safety,
    allocation=save_reserves,
    wind_fn=steady_west_wind,
    sun=overhead_sun,
)

t_late_storm = grow_tree(cfg, n_generations=n_generations, seed=seed, wind_fn=storm_at_gen_50)

print(f"{'scenario':>24}  {'branches':>10}  {'leaves':>8}  {'trunk d':>8}")
for label, tree in [
    ("default storm", t_default),
    ("steady-west + cautious", t_steady),
    ("calm-then-storm", t_late_storm),
]:
    print(
        f"{label:>24}  "
        f"{tree.get_number_of_branches():>10}  "
        f"{tree.get_total_leaves():>8}  "
        f"{tree.get_diameter(0):>8.4f}"
    )

Read the numbers as a small parameter study:

- The **steady-west** tree should be visibly leaning eastward — every
  branch has felt the same drag in the same direction for 80 steps.
- The **calm-then-storm** tree has time to grow tall before the wind
  arrives, so the storm at gen 50 prunes more aggressively than in the
  default run.
- The **default** run is the noisiest because the wind direction
  rotates.

In [ ]:
scenarios = [
    ("default storm", t_default),
    ("steady-west + cautious", t_steady),
    ("calm-then-storm", t_late_storm),
]
fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "scene"} for _ in scenarios]],
    subplot_titles=[label for label, _ in scenarios],
)
for col, (_, tree) in enumerate(scenarios, start=1):
    for trace in plot_tree_3d(tree).data:
        fig.add_trace(trace, row=1, col=col)
fig.update_layout(width=1500, height=550, paper_bgcolor="white", showlegend=False)
fig.show()

## Where to next

- Combine the custom wind with the neural genome from
  [03_neural_genome.ipynb](03_neural_genome.ipynb).
- [05_strahler_diagnostics.ipynb](05_strahler_diagnostics.ipynb) —
  quantify the differences with self-similarity diagnostics.
- For more sophisticated control over the safety/allocation rules, the
  upcoming `SymPySafety` / `SymPyAllocation` will let you write the
  decision as a closed-form expression in the YAML.